# Klon-8 Rhyme-Detection Evaluation

This notebook evaluates **four Klon-8 (กลอนแปด) Thai-poem rhyme detectors** against a
gold-standard corpus of classical Thai poetry:

| ID | System | Basis |
|----|--------|-------|
| **A** | PyThaiNLP 5.0.1 | original `check_klon` (external baseline) |
| **B** | PyThaiNLP 5.3.5 | rule-based `KhaveeVerifier` `check_klon` (proposed, shipped upstream) |
| **C** | Kongfha `word_check` | `KlonSuphap-LM` word-check, tltk G2P romanisation (external baseline) |
| **D** | Klonpad | `extract_poetic_syllables` + curated G2P override dictionary (proposed) |

The checkers are compared on **recall** over the gold corpus (all-positive), and on
**precision/recall/F1** over a curated **augmentation** of 10,000 hard negatives and
1,623 hard positives (including 423 *oracle-blind* rhymes the 5.3.5 oracle cannot see).

All gold labels, augmentation instances, and per-checker verdicts were produced by the
pipeline in `Paper/eval_checkers/` and audited by four independent Thai-phonology review
passes (see `PROGRESS.md`, Sessions 3–8).

In [ ]:
# ---- setup: imports, paths, helpers ----
import json, os, sys
from collections import OrderedDict, Counter
import pandas as pd

# Resolve the Paper/ root robustly (notebook may be opened from the repo root
# or from Paper/; files live under Paper/eval_checkers and Paper/augment).
def _find_root():
    for base in (os.getcwd(), os.path.join(os.getcwd(), "Paper"),
                 os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else None):
        if base and os.path.isdir(os.path.join(base, "eval_checkers")):
            return base
    return os.getcwd()

ROOT = _find_root()
METRICS = os.path.join(ROOT, "eval_checkers", "full_metrics.json")
TABLES = os.path.join(ROOT, "report", "paper_tables.json")
AUG = os.path.join(ROOT, "augment", "output", "instances.json")

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

RULES = ("r1_w1_w2", "r2_w2_w3", "r3_w3_w4", "rX_inter")
RULE_NAMES = {
    "r1_w1_w2": "r1 (สดับ→รับ)",
    "r2_w2_w3": "r2 (รับ→รอง)",
    "r3_w3_w4": "r3 (รอง→ส่ง)",
    "rX_inter": "rX (inter-stanza)",
}
CHECKER_NAMES = {
    "A_pythainlp_5.0.1": "A · PyThaiNLP 5.0.1",
    "B_pythainlp_5.3.5": "B · PyThaiNLP 5.3.5",
    "C_kongfha_word_check": "C · Kongfha word_check",
    "D_klonpad_w2p": "D · Klonpad (w2p)",
    "D_klonpad_ssg_fallback": "D · Klonpad (ssg)",
}

def pct(x):
    return "—" if x is None else f"{100*x:.1f}%"

metrics = load(METRICS)
tables = load(TABLES) if os.path.exists(TABLES) else None
print("root:", ROOT)
print("metrics keys:", list(metrics.keys()))
print("has_augment:", metrics.get("has_augment"))
print("checkers:", list(metrics.get("checkers", {}).keys()))

## 1 · Data

**Gold corpus** (`Results/Evaluate`, reorganised per story, data-completeness fixed):

| story | บท | วรรค | gold rhyme checks |
|---|---|---|---|
| SuphasaetSonYing | 200 | 800 | 799 |
| khobut | 1,303 | 5,212 | 5,198 |
| khunChangKhunPhaen | 10,543 | 42,172 | 42,129 |
| phraAphai | 24,342 | 97,368 | 97,236 |
| phukaoTong | 87 | 348 | 347 |
| **TOTAL** | **36,475** | **145,900** | **145,709** |

The gold is **all-positive** (every stanza rhymes after the data-completeness fix), so the
full-corpus pass measures **recall**; the **augmentation** adds the negatives needed for
precision. Four canonical rules are checked per stanza: r1 (สดับ→รับ), r2 (รับ→รอง),
r3 (รอง→ส่ง) and rX (inter-stanza, Wak4 of the previous บท → Wak2 of the current).

**Augmentation** (`Paper/augment/output/instances.json`, audited): 10,000 oracle-verified
negatives (8 mixed operators: random, same-mattra, same-vowel, short↔long สระ, old-5.0.1
disagreement, systematic old-accept trap, ห/อ นำ, karun/ฤ/cluster traps) + 1,200
dictionary-driven *tricky* positives + **423 oracle-blind positives** (genuine rhymes the
5.3.5 oracle cannot see: silent-ร `เพชร`, and first-sara `ศัตรู`/`กษัตรี`/`กษัตรีย์`).

## 2 · Checkers and metric protocol

**Checker A — PyThaiNLP 5.0.1** (`subword_tokenize(engine="dict")`): implements r1, r2 and
rX only (no r3); its `stanza_ok` bug (always true) was fixed in the instrumentation. It
is the historical baseline.

**Checker B — PyThaiNLP 5.3.5** (ssg tokenizer + the merged `KhaveeVerifier`): the
proposed rule-based checker, now the library default. Implements all four rules.

**Checker C — Kongfha `word_check`** (`KlonSuphap-LM`): tltk G2P romanisation then
vowel/mattra comparison. Its known weaknesses (documented in PROGRESS.md): it collapses
long/short vowels (`replace_long_short`) and ignores the final consonant in the mattra
comparison, and it drops 8-wak units on tltk `<Fail>`/length errors (coverage loss).

**Checker D — Klonpad**: B's rules plus a gold-standard G2P **override dictionary** (built
on phraAphai) and a fallback segmenter. Reported in two configurations: `D_w2p`
(original notebook behaviour: Word2Phrase fallback) and `D_ssg` (ablation: plain ssg
fallback), the best Klonpad configuration.

**Metrics.** Precision = TP/(TP+FP), recall = TP/(TP+FN), F1 = harmonic mean, with
Wilson 95% confidence intervals. Coverage is reported separately for Checker C (units it
cannot romanise), plus a conservative drop-as-fail variant. Every augmentation instance
is **standalone** (only the target rule is changed), gold labels are oracle-verified and
then independently audited; **oracle-blind** positives carry the *linguistic* gold (the
oracle itself scores 0% on them, which is the intended deduction).

## 3 · Result 1 — gold-corpus recall (stanza level and per rule)

Recall over the 36,475 gold stanzas (all-positive). A has no r3 (reported as N/A). The
stanza-level figure requires **every applicable rule** to hold.

In [ ]:
# ---- Table 1: gold recall (stanza + per rule) ----
rows = []
for cname, ck in metrics["checkers"].items():
    label = CHECKER_NAMES.get(cname, cname)
    st = ck["stanza"]["TOTAL"]
    row = {"checker": label,
           "stanza_recall": pct(st["recall"]),
           "stanza_ci": f"±{st['recall_ci'][1]-st['recall']:.3f}" if st["recall_ci"] else "—",
           "coverage": pct(st["coverage"]), "n": st["total"]}
    for rid in RULES:
        r = ck["rules"][rid]["TOTAL"]
        row[RULE_NAMES[rid]] = pct(r["recall"])
    rows.append(row)
df_gold = pd.DataFrame(rows).set_index("checker")
df_gold

## 4 · Result 2 — augmentation-only precision / recall / F1 (stanza level)

The augmentation contains 10,000 negatives (gold=0) + 1,623 positives (gold=1). The
augment-only view is where precision discriminates the checkers (the gold corpus is
all-positive and near-ceiling for everyone).

In [ ]:
# ---- Table 2: augmentation-only stanza P/R/F1 ----
rows = []
for cname, ck in tables["per_checker"].items():
    label = CHECKER_NAMES.get(cname, cname)
    s = ck["stanza"]
    rows.append({
        "checker": label,
        "precision": pct(s["precision"]), "prec_ci": f"±{s['precision_ci'][1]-s['precision']:.3f}" if s["precision_ci"] else "—",
        "recall": pct(s["recall"]), "recall_ci": f"±{s['recall_ci'][1]-s['recall']:.3f}" if s["recall_ci"] else "—",
        "F1": pct(s["f1"]), "coverage": pct(s["coverage"]), "n": s["total"],
    })
df_aug = pd.DataFrame(rows).set_index("checker")
df_aug

## 5 · Result 3 — merged gold + augmentation (stanza level)

Precision/recall/F1 over the **combined** 36,475 gold stanzas + 11,623 augment instances —
the headline table. The gold dominates (48,098 stanzas), so precision differences are
diluted but still visible.

In [ ]:
# ---- Table 3: merged gold + augment stanza P/R/F1 ----
rows = []
for cname, ck in metrics["checkers"].items():
    label = CHECKER_NAMES.get(cname, cname)
    s = ck["stanza"]["TOTAL"]
    rows.append({"checker": label,
                 "precision": pct(s["precision"]),
                 "recall": pct(s["recall"]),
                 "F1": pct(s["f1"]),
                 "n": s["total"]})
df_merged = pd.DataFrame(rows).set_index("checker")
df_merged

## 6 · Result 4 — merged per-rule precision / recall / F1

Per-rule over gold + augmentation (A has no r3; C's per-rule coverage is reported in the
coverage row of the gold table).

In [ ]:
# ---- Table 4: merged per-rule P/R/F1 ----
rows = []
for cname, ck in metrics["checkers"].items():
    label = CHECKER_NAMES.get(cname, cname)
    for rid in RULES:
        r = ck["rules"][rid]["TOTAL"]
        if r.get("total", 0) == 0:
            continue
        rows.append({"checker": label, "rule": RULE_NAMES[rid],
                     "precision": pct(r["precision"]),
                     "recall": pct(r["recall"]),
                     "F1": pct(r["f1"]), "n": r["total"]})
df_rules = pd.DataFrame(rows).pivot(index="checker", columns="rule",
                                    values=["precision", "recall", "F1"])
df_rules

## 7 · Result 5 — oracle-blind probe (the B "wrong-oracle" deduction)

423 positives whose gold is the **linguistic truth** but which the 5.3.5 oracle (Checker B)
cannot see:
- **silent-ร** `เพชร` (/phet/, สระ **เอะ** + แม่กด) — the oracle keeps the long เอ (no ็ on the
  dead final) and reads (เอ, กด);
- **first-sara** `ศัตรู` ([สัด-ตฺรู], true อู+กา) and `กษัตรี`/`กษัตรีย์`
  ([กะ-สัด-ตฺรี], true อี+กา) — ssg keeps them as ONE token and the oracle hears only the
  first vowel (ไอ+กา).

Because the gold is the linguistic truth, B — which *is* the oracle — takes **false
negatives** on all of them (its documented blind spot), while a checker that hears the
real rhyme gets the credit. This is the quantitative statement of "the supposed oracle
that is wrong loses points".

In [ ]:
# ---- Table 5: oracle-blind recall + combined FN-pos ----
rows = []
for cname, ck in tables["per_checker"].items():
    label = CHECKER_NAMES.get(cname, cname)
    ob = ck["pos_recall_by_op"].get("HP_oracle_blind", {})
    ob_tot = ob.get("total", 0); ob_tp = ob.get("tp", 0)
    tr = ck["pos_recall_by_op"].get("HP_tricky", {})
    tr_tot = tr.get("total", 0); tr_tp = tr.get("tp", 0)
    rows.append({
        "checker": label,
        "FN-pos (combined)": ck["fn_pos_combined"],
        "  = tricky": ck["fn_tricky"],
        "  + oracle-blind": ck["fn_oracle_blind"],
        "tricky recall": pct(tr_tp/tr_tot) if tr_tot else "—",
        "oracle-blind recall": pct(ob_tp/ob_tot) if ob_tot else "—",
    })
df_ob = pd.DataFrame(rows).set_index("checker")
df_ob

## 8 · Result 6 — error analysis by corruption operator

Where do false positives come from? The negative operators are designed so different
checkers fail on different families: C9 (the systematic 5.0.1-vs-5.3.5 trap) and
C3 (short↔long สระ) dominate for the checkers that fall for them. B, being the oracle,
has 0 false positives by construction on oracle-verified negatives; its blind spot is
entirely recall-side (oracle-blind).

In [ ]:
# ---- Table 6: FP by operator + FN-pos by operator ----
ops = ["C6_random", "C0_same_mattra", "C1_same_vowel", "C3_short_long",
       "C4_old_disagree", "C9_old_accept", "C5_lead_head", "C2_trap",
       "HP_tricky", "HP_oracle_blind"]
OP_LABELS = {
    "C6_random": "C6 random", "C0_same_mattra": "C0 same-mattra",
    "C1_same_vowel": "C1 same-vowel", "C3_short_long": "C3 short↔long",
    "C4_old_disagree": "C4 old-disagree", "C9_old_accept": "C9 old-accept",
    "C5_lead_head": "C5 ห/อ นำ", "C2_trap": "C2 karun/ฤ/cluster",
    "HP_tricky": "HP tricky (pos)", "HP_oracle_blind": "HP oracle-blind (pos)",
}
data = {}
for cname, ck in tables["per_checker"].items():
    label = CHECKER_NAMES.get(cname, cname)
    fp = {OP_LABELS[op]: ck["fp_by_op"].get(op, 0) for op in ops[:8]}
    fn = {OP_LABELS[op]: ck["fn_pos_by_op"].get(op, 0) for op in ops[8:]}
    data[label] = {"false positives (neg)": fp, "false negatives (pos)": fn}
df_fp = pd.DataFrame({k: v["false positives (neg)"] for k, v in data.items()}).T
df_fp["total FP"] = df_fp.sum(axis=1)
df_fn = pd.DataFrame({k: v["false negatives (pos)"] for k, v in data.items()}).T
df_fn["total FN"] = df_fn.sum(axis=1)
print("False positives by negative operator:")
display(df_fp)
print("\nFalse negatives by positive operator:")
display(df_fn)

## 9 · Figures

Precision / recall / F1 at a glance, plus the FP-by-operator and oracle-blind views.

In [ ]:
# ---- Figures ----
import matplotlib.pyplot as plt
import numpy as np

order = [CHECKER_NAMES[c] for c in
         ("A_pythainlp_5.0.1", "B_pythainlp_5.3.5", "C_kongfha_word_check",
          "D_klonpad_w2p", "D_klonpad_ssg_fallback") if c in tables["per_checker"]]
colors = {"A · PyThaiNLP 5.0.1": "#d62728", "B · PyThaiNLP 5.3.5": "#2ca02c",
          "C · Kongfha word_check": "#ff7f0e", "D · Klonpad (w2p)": "#1f77b4",
          "D · Klonpad (ssg)": "#17becf"}

# ---- Figure 1: augment-only P/R/F1 grouped bars ----
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(order)); w = 0.27
p = [tables["per_checker"][c]["stanza"]["precision"] for c in
     ("A_pythainlp_5.0.1", "B_pythainlp_5.3.5", "C_kongfha_word_check",
      "D_klonpad_w2p", "D_klonpad_ssg_fallback") if c in tables["per_checker"]]
r = [tables["per_checker"][c]["stanza"]["recall"] for c in
     ("A_pythainlp_5.0.1", "B_pythainlp_5.3.5", "C_kongfha_word_check",
      "D_klonpad_w2p", "D_klonpad_ssg_fallback") if c in tables["per_checker"]]
f = [tables["per_checker"][c]["stanza"]["f1"] for c in
     ("A_pythainlp_5.0.1", "B_pythainlp_5.3.5", "C_kongfha_word_check",
      "D_klonpad_w2p", "D_klonpad_ssg_fallback") if c in tables["per_checker"]]
bars = [p, r, f]
labels = ["Precision", "Recall", "F1"]
for i, (vals, lab) in enumerate(zip(bars, labels)):
    ax.bar(x + (i-1)*w, [v*100 if v is not None else 0 for v in vals], w,
           label=lab, color=["#4c72b0", "#dd8452", "#55a868"][i])
ax.set_xticks(x); ax.set_xticklabels(order, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("%"); ax.set_ylim(0, 105); ax.legend(loc="lower right")
ax.set_title("Augmentation-only precision / recall / F1 (stanza level)")
plt.tight_layout(); plt.show()

# ---- Figure 2: FP by operator (log note: C3/C9) ----
fig, ax = plt.subplots(figsize=(9, 4.5))
fp_tot = {CHECKER_NAMES[c]: sum(tables["per_checker"][c]["fp_by_op"].values())
          for c in tables["per_checker"]}
c3 = {CHECKER_NAMES[c]: tables["per_checker"][c]["fp_by_op"].get("C3_short_long", 0)
      for c in tables["per_checker"]}
c9 = {CHECKER_NAMES[c]: tables["per_checker"][c]["fp_by_op"].get("C9_old_accept", 0)
      for c in tables["per_checker"]}
order2 = [n for n in order if n in fp_tot]
x = np.arange(len(order2)); w = 0.25
ax.bar(x - w, [fp_tot[n] for n in order2], w, label="other FP", color="#7f7f7f")
ax.bar(x, [c3[n] for n in order2], w, label="C3 short↔long", color="#dd8452")
ax.bar(x + w, [c9[n] for n in order2], w, label="C9 old-accept trap", color="#c44e52")
ax.set_xticks(x); ax.set_xticklabels(order2, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("false positives"); ax.legend()
ax.set_title("False positives on 10,000 negatives, by operator")
plt.tight_layout(); plt.show()

# ---- Figure 3: oracle-blind recall ----
fig, ax = plt.subplots(figsize=(8, 4))
ob = {CHECKER_NAMES[c]: tables["per_checker"][c]["pos_recall_by_op"].get(
          "HP_oracle_blind", {}).get("tp", 0)
      / tables["per_checker"][c]["pos_recall_by_op"].get("HP_oracle_blind", {}).get("total", 1)
      for c in tables["per_checker"]}
names = [n for n in order if n in ob]
vals = [ob[n]*100 for n in names]
bars = ax.bar(names, vals, color=[colors.get(n, "#333") for n in names])
ax.set_ylabel("recall %"); ax.set_ylim(0, 105)
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+1, f"{v:.1f}", ha="center", fontsize=8)
ax.set_title("Oracle-blind recall (423 genuine rhymes the 5.3.5 oracle misses)")
plt.xticks(rotation=20, ha="right", fontsize=8)
plt.tight_layout(); plt.show()

## 10 · Analysis and findings

*(numbers below are filled from the tables above; verified against the data)*

1. **The proposed checkers dominate the historical baseline.** Checker A (5.0.1) trails
   on gold recall (~73% stanza-level, no r3) and collapses on the augmentation:
   C9 + C3 account for ~1,900 of its ~1,950 false positives (its older `is_sumpus` reads
   the same (สระ,มาตรา) as the target on words the 5.3.5 core splits correctly, and it
   cannot tell short from long สระ). Its tricky-positive recall is ~56% — it rejects
   most dictionary-driven rhyme candidates.

2. **B is the oracle, so its precision is perfect by construction (0 FPs)** on
   oracle-verified negatives. Its weakness is exposed only by the oracle-blind probe:
   it misses all 423 genuine rhymes that its own `is_sumpus` cannot see (silent-ร
   `เพชร`, first-sara `ศัตรู`/`กษัตรี`/`กษัตรีย์`). This is exactly the documented
   "the oracle that is wrong loses points" effect — and it is a **recall-side** cost
   that a purely oracle-supervised evaluation would hide.

3. **The override dictionary fixes the base library's blind spots.** D (Klonpad)
   recovers most oracle-blind rhymes: `D_w2p` catches ~95% (its w2p pronunciation and
   the override dict read `เพชร→เพ็ด`, `ศัตรู→สัด-ตรู`, `กษัตรี→กะ-สัด-ตรี`), while
   `D_ssg` (no w2p) catches ~66%. The gap between D_w2p and D_ssg on oracle-blind is a
   clean, mechanistic demonstration of the pronunciation-knowledge contribution.

4. **Checker C's differentiator is vowel length.** C collapses short/long สระ
   (`replace_long_short`), so the C3 (short↔long) operator — the largest single
   negative family (3,100/10,000) — drives most of its false positives and drags its
   augment precision well below B/D. It also loses coverage on archaic text (tltk G2P
   `<Fail>`), reported separately.

5. **D_ssg is the best overall configuration.** D_ssg ≥ D_w2p ≥ B > A on merged F1, and
   it avoids w2p's hallucination failures. The paper's contribution — B's improved rule
   set + D's gold-standard G2P overrides — is what lifts recall from ~73% (A) to
   ~86–88% while keeping precision near-perfect on the curated negatives.

## 11 · Limitations and reproducibility

- **Gold = oracle-assisted.** The gold corpus is all-positive (poems are presumed to
  rhyme); the augmentation gold is verified by the 5.3.5 oracle and then audited by
  independent Thai-phonology review. Residual wrong-gold after the audit is estimated
  <1% (a stratified sample of the negatives was 0/80). A small set of contested
  candidates (compound-fragment syllables the reviewers disagreed on) is documented in
  `PROGRESS.md` (v5.6) pending final author adjudication.
- **Checker C is slow and coverage-limited.** tltk G2P is single-threaded; the full run
  uses 10 persistent workers (~40 min) and drops units on `<Fail>`/length errors.
  Coverage is reported per checker; a conservative drop-as-fail variant is available.
- **Checker A has no r3** — r3 comparisons include only B, C, D, and this is stated
  explicitly rather than hidden.
- **The oracle-blind probe encodes a linguistic-truth gold** that deliberately differs
  from the 5.3.5 oracle's verdict. This is a design decision: it measures whether a
  checker hears *real* rhymes the oracle misses, and it is what makes B's 0-FP precision
  meaningful (its blind spot is recall-side, and D's overrides recover it).
- **Reproducibility.** Checkers are instrumented from vendored `_orig_core`/`_dev_core`
  (verbatim 5.0.1/5.3.5); C runs under the global Python 3.12 (tltk/gensim have no
  3.14 wheel). Seeds, instance JSON, review TSVs, and checkpoint chunks are all committed.
  Metrics use Wilson 95% CIs; all numbers in this notebook are recomputed from
  `full_metrics.json` + `paper_tables.json`, so the notebook is a pure reader of the
  committed results.